# Exogenous Regressors

> How to include exogenous regressors.

With `StatsForecast` you can easily include exogenous regressors. Some methods, such as `AutoARIMA`, have the ability to include exogenous regressors, while other models only use the time series information. `StatsForecast` takes care of passing the exogenous variables to the models that use them. 

## Install statsforecast

In [ ]:
%%capture
!pip install statsforecast

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA
from statsforecast.utils import AirPassengersDF as Y_df

In this example, we will use the `AirPassengers` dataset to show how to work with exogenous regressors.

In [ ]:
Y_df.head()

## Split train/test sets

We will use the last `12` observations of the dataset as the test set.

In [ ]:
horizon = 12
Y_train_df = Y_df[Y_df.ds<='1959-12-31'] # 132 train
Y_test_df = Y_df[Y_df.ds>'1959-12-31'] # 12 test

## Add exogenous regressors

In this example, we will include the `trend` as exogenous regressor. But you can include as many as you have or want.

In [ ]:
%%capture
Y_train_df['trend'] = np.arange(1, len(Y_train_df) + 1)

In [ ]:
Y_train_df.head()

In [ ]:
Y_train_df.tail()

Observe that the exogenous regressors have to be placed after the target variable `y`.

### Create future exogenous regressors

In order for the model to produce forecasts, it needs the future exogenous regressors. In this section we will construct a dataframe that includes the future `trend`.

In [ ]:
X_test_df = pd.DataFrame({
  'unique_id': 1.0,
  'ds': pd.date_range(start='1960-01-01', periods=horizon, freq='M')
})
# We construct xreg for test. The train series ends at the 133th step. 
X_test_df['trend'] = np.arange(133, 133 + horizon)
X_test_df.head()

## Train the model

In [ ]:
season_length = 12
model = StatsForecast(
    df=Y_train_df, 
    models=[AutoARIMA(season_length=12)], 
    freq='M', 
    n_jobs=-1
)

### Forecast mode

The `StatsForecast.forecast` method is more computationally efficient since it does not save objects during training and predicting. The method receives the future exogenous regressors `X_test_df`. 

In [ ]:
Y_hat_df = model.forecast(horizon, X_df=X_test_df)
Y_hat_df = Y_hat_df.reset_index()
Y_hat_df.head()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize = (20, 7))
Y_hat_df = Y_test_df.merge(Y_hat_df, how='left', on=['unique_id', 'ds'])
df_plot = pd.concat([Y_train_df, Y_hat_df_xreg]).set_index('ds')
df_plot[['y', 'AutoARIMA']].plot(ax=ax, linewidth=2)
ax.set_title('AirPassengers Forecast (with AutoARIMA external regressors)', fontsize=22)
ax.set_ylabel('Monthly Passengers', fontsize=20)
ax.set_xlabel('Timestamp [t]', fontsize=20)
ax.legend(prop={'size': 15})
ax.grid()
for label in (ax.get_xticklabels() + ax.get_yticklabels()):
    label.set_fontsize(20)

### SKlearn syntax

The `sklearn` syntax can also be used. Fist, train the model using the `StatsForecast.fit` method.

In [ ]:
model.fit()

The use the fitted model to produce forecasts. Observe that the `StatsForecast.predict` method receives the future exogenous regressors.

In [ ]:
model.predict(horizon, X_df=X_test_df)

## Including prediction intervals

You can also compute prediction intervals using exogenous regressors. Simply add the `level` argument.

### Forecast mode

In [ ]:
Y_hat_df = model.forecast(horizon, X_df=X_test_df, level=(80,95))
Y_hat_df = Y_hat_df.reset_index()
Y_hat_df.head()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize = (20, 7))
df_plot = pd.concat([Y_train_df, Y_hat_df]).set_index('ds')
df_plot[['y', 'AutoARIMA']].plot(ax=ax, linewidth=2)
ax.fill_between(df_plot.index, 
                df_plot['AutoARIMA-lo-80'], 
                df_plot['AutoARIMA-hi-80'],
                alpha=.35,
                color='orange',
                label='auto_arima_level_80')
ax.fill_between(df_plot.index, 
                df_plot['AutoARIMA-lo-95'], 
                df_plot['AutoARIMA-hi-95'],
                alpha=.2,
                color='orange',
                label='auto_arima_level_95')
ax.set_title('AirPassengers Forecast (with AutoARIMA external regressors and intervals)', fontsize=22)
ax.set_ylabel('Monthly Passengers', fontsize=20)
ax.set_xlabel('Timestamp [t]', fontsize=20)
ax.legend(prop={'size': 15})
ax.grid()
for label in (ax.get_xticklabels() + ax.get_yticklabels()):
    label.set_fontsize(20)

### SKlearn syntax

Since the model is already fitted, just add `level=(80,90)` to the `StatsForecast.predict` method.

In [ ]:
model.predict(horizon, X_df=X_test_df, level=(80, 90))